<a href="https://colab.research.google.com/github/avocado-planet/01-ReAct-Agent-Sample/blob/main/ReAct_Agent_Sample_ClassicStyle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# LangChain 0.2〜0.3 の場合
!pip install -q "langchain>=0.2,<1.0" langchain-community langchain-openai \
            langsmith google-search-results

In [2]:
"""
==============================================================================
【旧式】LangChain Classic ReAct Agent（テキストパース方式 / 第1世代）
==============================================================================

■ 方式の特徴:
  - LangChain の AgentExecutor + create_react_agent（第1世代）を使用
  - hwchase17/react プロンプトの {tools}, {tool_names}, {input}, {agent_scratchpad}
    がすべて正しく展開される
  - LLM はプレーンテキストで "Action:", "Action Input:" を出力
  - AgentExecutor がその出力を正規表現でパースしてツールを実行
  - ツール結果を {agent_scratchpad} に追記して再度 LLM を呼ぶ

■ 世代の位置づけ:
  【第1世代】AgentExecutor + create_react_agent（本コード）→ 非推奨 ★
  【第2世代】langgraph.prebuilt.create_react_agent（元のサンプル）→ 非推奨
  【第3世代】langchain.agents.create_agent → 現在の推奨

■ プロンプトの {} が埋まる仕組み:
  ┌─────────────────────┬────────────────────┬──────────────────────────────────┐
  │ プレースホルダー     │ 埋まるタイミング   │ 埋まる内容の例                    │
  ├─────────────────────┼────────────────────┼──────────────────────────────────┤
  │ {tools}             │ create_react_agent │ "Search: Useful for answering..." │
  │                     │ 呼び出し時         │ （ツール名: description の一覧）   │
  ├─────────────────────┼────────────────────┼──────────────────────────────────┤
  │ {tool_names}        │ create_react_agent │ "Search"                          │
  │                     │ 呼び出し時         │ （ツール名のカンマ区切り）         │
  ├─────────────────────┼────────────────────┼──────────────────────────────────┤
  │ {input}             │ invoke() 実行時    │ "2026/4/1ドジャーズの先発投手は？" │
  │                     │                    │ （ユーザーの質問文そのまま）       │
  ├─────────────────────┼────────────────────┼──────────────────────────────────┤
  │ {agent_scratchpad}  │ ループの各ステップ │ 初回: ""（空文字列）               │
  │                     │ で自動更新         │ 2回目以降:                        │
  │                     │                    │   Action: Search                  │
  │                     │                    │   Action Input: "..."             │
  │                     │                    │   Observation: 検索結果...         │
  │                     │                    │   Thought:                        │
  └─────────────────────┴────────────────────┴──────────────────────────────────┘

■ verbose=True 時のコンソール出力例:
  > Entering new AgentExecutor chain...
  Thought: 最新のスポーツ情報なので検索が必要だ
  Action: Search
  Action Input: "2026年4月1日 ドジャーズ 先発投手"
  Observation: 4月1日のドジャーズ対パドレス戦の先発は...
  Thought: I now know the final answer
  Final Answer: 2026年4月1日のドジャーズの先発投手は○○です。
  > Finished chain.

■ 非推奨情報（2026年4月時点）:
  - AgentExecutor は LangChain 0.2 で非推奨
  - LangChain 1.0 では langchain-classic パッケージに移動
  - 新規採用は推奨されない（学習目的でのみ使用）

■ 必要パッケージ:
  【LangChain 0.2〜0.3（1.0より前）の場合】
    pip install "langchain>=0.2,<1.0" langchain-community langchain-openai \
                langsmith google-search-results

  【LangChain 1.0 以降の場合】
    pip install "langchain>=1.0" langchain-classic langchain-community \
                langchain-openai langsmith google-search-results

  ★ google-search-results は SerpAPIWrapper の内部依存パッケージ
    （忘れやすいので注意）

■ 環境変数:
  OPENAI_API_KEY     : OpenAI API キー（必須）
  SERPAPI_API_KEY     : SerpAPI キー（必須）
  LANGSMITH_API_KEY   : LangSmith キー（オプション、プロンプトHub / トレース用）
  LANGCHAIN_TRACING_V2: "true" でトレース有効（オプション）
==============================================================================
"""

import os
import sys
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["SERPAPI_API_KEY"] = userdata.get("SERPAPI_API_KEY")

# ---------------------------------------------------------------------------
# 1. 環境変数チェック
# ---------------------------------------------------------------------------
required_env_vars = ["OPENAI_API_KEY", "SERPAPI_API_KEY"]
missing = [v for v in required_env_vars if not os.environ.get(v)]
if missing:
    print(f"[ERROR] 以下の環境変数が未設定です: {', '.join(missing)}")
    print("  例: export OPENAI_API_KEY='sk-...'")
    print("  例: export SERPAPI_API_KEY='...'")
    sys.exit(1)

# ---------------------------------------------------------------------------
# 2. ライブラリのインポート
#    LangChain 1.0 では AgentExecutor が langchain-classic に移動した。
#    どちらのバージョンでも動くように自動判定する。
# ---------------------------------------------------------------------------
try:
    # LangChain 1.0+ の場合: langchain-classic からインポート
    from langchain_classic.agents import AgentExecutor, create_react_agent
    print("[INFO] langchain-classic からインポート（LangChain 1.0+）")
except ImportError:
    try:
        # LangChain 0.2〜0.3 の場合
        from langchain.agents import AgentExecutor, create_react_agent
        print("[INFO] langchain.agents からインポート（LangChain <1.0）")
    except ImportError:
        print("[ERROR] AgentExecutor が見つかりません。")
        print("  LangChain 1.0+ → pip install langchain-classic")
        print("  LangChain <1.0 → pip install 'langchain>=0.2,<1.0'")
        sys.exit(1)

from langchain_core.tools import Tool, render_text_description
from langchain_core.prompts import PromptTemplate
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import ChatOpenAI
from langsmith import Client

[INFO] langchain.agents からインポート（LangChain <1.0）


In [3]:
# ---------------------------------------------------------------------------
# 3. LLM の設定
#    - temperature=0: 出力の再現性を高める
#    - model: gpt-4o-mini（gpt-3.5-turbo は2025年以降非推奨）
# ---------------------------------------------------------------------------
llm = ChatOpenAI(
    temperature=0,
    model="gpt-4o-mini",
)

# ---------------------------------------------------------------------------
# 4. ツールの設定
#    ★ 旧式では description がプロンプトの {tools} にテキストとして埋め込まれる
#      description が曖昧だと LLM がツールを適切に選べない
# ---------------------------------------------------------------------------
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description=(
            "Useful for answering questions about current events, "
            "recent news, or real-time information that the LLM may not know."
        ),
    )
]

# ---------------------------------------------------------------------------
# 5. プロンプトテンプレートの取得
#
#    hwchase17/react プロンプトには以下の4つのプレースホルダーがある:
#      {tools}            → ツールの名前と説明の一覧
#      {tool_names}       → ツール名のカンマ区切りリスト
#      {input}            → ユーザーの質問文
#      {agent_scratchpad} → 中間ステップの履歴（Thought/Action/Observation）
#
#    ★ 旧式 create_react_agent はこれらを正しく展開する。
#    ★ 新式（create_agent）では {tools} 等は使わない。
# ---------------------------------------------------------------------------

# 方法1: LangSmith Hub から取得
try:
    client = Client()
    prompt = client.pull_prompt("hwchase17/react")
    print("\n[INFO] LangSmith Hub からプロンプトを取得しました")
except Exception as e:
    print(f"\n[WARN] LangSmith Hub 取得失敗: {e}")
    print("[INFO] ハードコードされたプロンプトを使用します")

    # 方法2: ハードコード（hwchase17/react の実際の内容）
    template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

    prompt = PromptTemplate(
        template=template,
        input_variables=["tools", "tool_names", "input", "agent_scratchpad"],
    )



[INFO] LangSmith Hub からプロンプトを取得しました


In [4]:
# ---------------------------------------------------------------------------
# 6. プロンプトの展開内容を確認（デバッグ用）
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("【デバッグ】プロンプトテンプレート（展開前）")
print("=" * 70)
raw_template = prompt.template if hasattr(prompt, "template") else str(prompt)
print(raw_template)

tools_text = render_text_description(tools)
tool_names_text = ", ".join([t.name for t in tools])

print("\n" + "=" * 70)
print("【デバッグ】{tools} に埋まる内容:")
print("=" * 70)
print(tools_text)

print("\n" + "=" * 70)
print("【デバッグ】{tool_names} に埋まる内容:")
print("=" * 70)
print(tool_names_text)

print("\n" + "=" * 70)
print("【デバッグ】{input} に埋まる内容:")
print("=" * 70)
print("→ invoke() に渡す質問文がそのまま入る")

print("\n" + "=" * 70)
print("【デバッグ】{agent_scratchpad} に埋まる内容:")
print("=" * 70)
print("→ 初回: 空文字列（何もない状態からスタート）")
print("→ 2回目以降（例）:")
print('   Action: Search')
print('   Action Input: "2026/4/1 Dodgers starting pitcher"')
print('   Observation: The starting pitcher for the Dodgers on April 1...')
print('   Thought:')


【デバッグ】プロンプトテンプレート（展開前）
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}

【デバッグ】{tools} に埋まる内容:
Search(query: str, **kwargs: Any) -> str - Useful for answering questions about current events, recent news, or real-time information that the LLM may not know.

【デバッグ】{tool_names} に埋まる内容:
Search

【デバッグ】{input} に埋まる内容:
→ invoke() に渡す質問文がそのまま入る

【デバッグ】{agent_scratchpad} に埋まる内容:
→ 初回: 空文字列（何もない状態からスタート）
→ 2回目以降（例）:
   Action: Search
   Action Input: "2026/4/1 Dodgers starti

In [5]:
# ---------------------------------------------------------------------------
# 7. エージェントの作成
#
#    旧式 create_react_agent が行うこと:
#      (a) prompt の {tools} にツール説明テキストを埋め込む（部分適用）
#      (b) prompt の {tool_names} にツール名リストを埋め込む（部分適用）
#      (c) LLM の出力テキストから "Action:" と "Action Input:" を
#          正規表現でパースするパーサーを組み込む
#
#    ※ {input} と {agent_scratchpad} は実行時に埋まる
#    ※ agent 単体は「パーサー付きの Runnable」
#      実行ループ（Thought→Action→Observation の繰り返し）は
#      AgentExecutor が担当する
# ---------------------------------------------------------------------------
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

# ---------------------------------------------------------------------------
# 8. AgentExecutor の作成（実行ループの管理）
#
#    AgentExecutor が担うこと:
#      1. {input} にユーザーの質問を埋めて LLM を呼ぶ
#      2. LLM のテキスト出力から "Action:" と "Action Input:" をパース
#      3. 該当ツールを実行し、結果を "Observation:" として取得
#      4. {agent_scratchpad} に Thought/Action/Observation を追記
#      5. "Final Answer:" が出るまで 1〜4 を繰り返す
#
#    パラメータ:
#    - verbose=True          : Thought/Action/Observation の全過程を表示 ★
#    - handle_parsing_errors : LLM 出力がフォーマットに従わない場合リトライ
#    - max_iterations=5      : 無限ループ防止（最大5回のツール呼び出し）
# ---------------------------------------------------------------------------
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5,
)

In [7]:
# ---------------------------------------------------------------------------
# 9. 実行
#    ★ 旧式は "input" キーで質問を渡す（新式の "messages" とは異なる）
#
#    実行時に起こること:
#
#    【ステップ1: 初回 LLM 呼び出し】
#      展開後のプロンプト:
#        Answer the following questions as best you can.
#        You have access to the following tools:
#        Search: Useful for answering questions about current events...  ← {tools}
#        ...
#        Action: should be one of [Search]                              ← {tool_names}
#        ...
#        Question: 2026/4/1ドジャーズの先発投手は？                      ← {input}
#        Thought:                                                       ← {agent_scratchpad}（空）
#
#      LLM の出力（プレーンテキスト）:
#        I need to search for current sports information.
#        Action: Search
#        Action Input: "2026年4月1日 ドジャーズ 先発投手"
#
#    【ステップ2: パース＋ツール実行】
#      正規表現で "Action: Search" と "Action Input: ..." を抽出
#      → SerpAPI 実行 → 結果取得
#
#    【ステップ3: 2回目 LLM 呼び出し】
#      {agent_scratchpad} に中間結果が追記された状態:
#        ...
#        Question: 2026/4/1ドジャーズの先発投手は？
#        Thought: I need to search for current sports information.
#        Action: Search                                                 ← {agent_scratchpad}
#        Action Input: "2026年4月1日 ドジャーズ 先発投手"                  に蓄積された
#        Observation: 4月1日のドジャーズ対パドレス戦の先発は○○...         中間結果
#        Thought:
#
#      LLM の出力:
#        I now know the final answer
#        Final Answer: 2026年4月1日のドジャーズの先発投手は○○です。
#
#    【終了】
#      "Final Answer:" を検出 → ループ終了 → 回答を返す
# ---------------------------------------------------------------------------
question = "日本時間2026/4/1ドジャーズの先発投手は？"

print("\n" + "=" * 70)
print(f"質問: {question}")
print("=" * 70)

try:
    # ★ 旧式は {"input": question}、新式は {"messages": [...]}
    result = agent_executor.invoke({"input": question})

    print("\n" + "=" * 70)
    print("最終回答:")
    print("=" * 70)
    # ★ 旧式は result["output"]、新式は result["messages"]
    print(result["output"])

except Exception as e:
    print(f"\n[ERROR] エージェント実行中にエラーが発生しました: {e}")
    print("  - APIキーが正しいか確認してください")
    print("  - ネットワーク接続を確認してください")
    raise


質問: 日本時間2026/4/1ドジャーズの先発投手は？


> Entering new AgentExecutor chain...
この質問は未来の特定の日におけるドジャーズの先発投手について尋ねていますが、現時点ではその情報はわかりません。2026年の先発投手はシーズンの進行や選手の状況によって変わる可能性があるため、最新の情報を確認する必要があります。したがって、ドジャーズの2026年4月1日の先発投手について調べるために、検索を行います。

Action: Search
Action Input: "2026年4月1日 ドジャーズ 先発投手"
['勝利投手. ドジャース. 大谷 (1勝0敗0S). 敗戦投手. ガーディアンズ. T ... 大谷 翔平, 先発, タナー・バイビー. 指, 大谷 翔平, 1, 中, スティーブン ...', '大谷翔平は4月1日のガーディアンズ戦で「1番・投手」として先発し、5回1安打無失点と圧巻の内容を披露。中5日〜6日の間隔での登板が想定されており ...', '2026年ドジャースの先発ローテーション候補 ; 大谷翔平, 14試 1勝1敗 防2.87 ; スネル, 11試 5勝4敗 防2.35 ; グラスノー, 18試 4勝3敗 防3.19 ; シーアン, 15試 6勝3敗 防2.82.', '投手としての初登板は日本時間4月1日のクリーブランド・ガーディアンズ戦 ... なお佐々木は開幕ローテーションの一角として、3月31日の開幕4戦目に先発登板 ...', '2026年シーズンのドジャースは、山本由伸、大谷翔平、佐々木朗希の3人が開幕時点の先発ローテーションを構成する見通しとなり、大きな注目を集めています ...', 'ドジャース ; (指), 大谷 翔平 .200 ; (右), カイル・タッカー .211 ; (遊), ムーキー・ベッツ .158 ; (一), フレディ・フリーマン .200 ...', '先発ピッチャーはガーディアンズがバイビー、ドジャースが大谷 翔平. 先攻:ガーディアンズのスターティングメンバーは1番:クワン（中）、2番 ...', 'MLBドジャース vs. ガーディアンズ戦！大谷翔平さんが先発投手で1番DH出場で、好投を見せてくれています！ ホームランもそろそろ見たい ...', 'ド